In [5]:
from pyspark.sql.functions import col, to_date, trim, current_timestamp, regexp_replace, expr, upper, to_timestamp

# 1. Clean Customers Data
df_cust = spark.read.table("bronze_customers") \
    .withColumn("customer_id", col("customer_id").cast("integer")) \
    .withColumn("customer_name", trim(col("customer_name"))) \
    .withColumn("city", trim(col("CITY"))) \
    .withColumn("customer_type", trim(col("CUSTOMER_TYPE"))) \
    .withColumn("registered_on", to_date(col("REGISTERED_ON"), "yyyy-MM-dd")) \
    .withColumn("kyc_verified_on", to_date(col("KYC_VERIFIED_ON"), "yyyy-MM-dd")) \
    .withColumn("_cleansed_at", current_timestamp())

df_cust.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_customers")
print(f"silver_customers created with {df_cust.count()} rows.")

# 2. Clean Depots Data
df_depots = spark.read.table("bronze_depots") \
    .withColumn("depot_id", trim(col("DEPOT_CODE"))) \
    .withColumn("depot_name", trim(col("DEPOT_NAME"))) \
    .withColumn("region", trim(col("ZONE"))) \
    .withColumn("fleet_size", col("FLEET_SIZE").cast("integer")) \
    .withColumn("_cleansed_at", current_timestamp())

df_depots.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_depots")
print(f"silver_depots created with {df_depots.count()} rows.")

# 3. Clean Rentals Data
# 3. Clean Rentals Data (FIXED: Correct lowercase column names & timestamp parsing)
df_rentals = spark.read.table("bronze_rentals") \
    .withColumn("rental_id", trim(col("rental_id"))) \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("depot_id", upper(trim(col("depot_code")))) \
    .withColumn("rental_start_date", to_timestamp(col("checkout_ts")).cast("date")) \
    .withColumn("rental_end_date", to_timestamp(col("checkin_ts")).cast("date")) \
    .withColumn("_cleansed_at", current_timestamp())

df_rentals.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_rentals")
print(f"silver_rentals created with {df_rentals.count()} rows.")

# 4. Clean Billing Data (FIXED: Upper-cased IDs & robust single-pass regex extraction for AMOUNT_INR)
df_billing = spark.read.table("bronze_billing") \
    .withColumn("bill_id", col("BILL_ID")) \
    .withColumn("rental_id", col("RENTAL_ID")) \
    .withColumn("bill_amount", 
        regexp_replace(
            expr("regexp_extract(AMOUNT_INR, '([0-9,]+\\\\.[0-9]+|[0-9]+)', 0)"), 
            ",", ""
        ).cast("double")
    ) \
    .withColumn("payer_type", trim(col("PAYER_TYPE"))) \
    .withColumn("billing_date", to_date(col("BILL_DATE"), "yyyy-MM-dd")) \
    .withColumn("_cleansed_at", current_timestamp())

df_billing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_billing")
print(f"silver_billing created with {df_billing.count()} rows.")

StatementMeta(, 96b45c70-78d0-4046-a198-7e3523a357c8, 7, Finished, Available, Finished, False)

silver_customers created with 602 rows.
silver_depots created with 6 rows.
silver_rentals created with 976 rows.
silver_billing created with 787 rows.


In [6]:
df = spark.sql("SELECT * FROM lh_buildmate.dbo.silver_rentals LIMIT 1000")
display(df)

StatementMeta(, 96b45c70-78d0-4046-a198-7e3523a357c8, 8, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 0f467490-0925-45d3-a5f5-f9e04921b6e7)

In [4]:
df = spark.sql("SELECT * FROM lh_buildmate.dbo.silver_depots LIMIT 1000")
display(df)

StatementMeta(, 96b45c70-78d0-4046-a198-7e3523a357c8, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0019ce43-c164-4e4b-a89a-25842eb7c809)

In [1]:
spark.read.table("bronze_rentals").printSchema()

StatementMeta(, 96b45c70-78d0-4046-a198-7e3523a357c8, 3, Finished, Available, Finished, False)

root
 |-- rental_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- depot_code: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- checkout_ts: string (nullable = true)
 |-- checkin_ts: string (nullable = true)
 |-- rental_type: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

